# Oscar — cleaning the Taiwan data

28 Sep 2026. Applies the agreed changes from `01-working` (discrepancies table)
and `02-duplicates`, and saves the result to
`data/processed/taiwan-clean.csv`. **Use that file from now on** instead of
the raw `.xls`.

`data/raw` is never edited. `data/processed` is not committed, so to get the
clean file, run this notebook top to bottom.

Order: remove duplicates first (the duplicate analysis used the raw codes),
then recode. Recoding first could turn near-duplicates into exact ones.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

taiwan = pd.read_excel(RAW / "taiwan-credit-default" / "default of credit card clients.xls",
                       header=1, index_col="ID")
Y = "default payment next month"
COLS = list(taiwan.columns)
taiwan.shape

(30000, 24)

## Step 1: drop the 5 "same every month" duplicates

From `02-duplicates`: these 5 pairs have the same non-zero bill and payment
every month. Fewer than 0.01 such pairs are expected by chance, so they are
treated as the same customer recorded twice. We keep the lower ID and drop the
other one. (`duplicated()` marks the *later* copy.)

The 30 no-activity pairs stay, because about half of them are expected by
chance and we can't tell which ones are real.

In [2]:
BILLS = [c for c in COLS if c.startswith("BILL_AMT")]
PAYS  = [c for c in COLS if c.startswith("PAY_AMT")]
MONEY = BILLS + PAYS

no_activity = (taiwan[MONEY] == 0).all(axis=1)
same_every_month = ((taiwan[BILLS].nunique(axis=1) == 1)
                    & (taiwan[PAYS].nunique(axis=1) == 1) & ~no_activity)

to_drop = taiwan.index[taiwan.duplicated() & same_every_month]
print("IDs dropped:", list(to_drop))
assert len(to_drop) == 5

clean = taiwan.drop(index=to_drop)
clean.shape

IDs dropped: [14295, 20876, 21882, 27352, 29266]


(29995, 24)

## Step 2: rename `PAY_0` to `PAY_1`

The paper's `X6`...`X11` cover all six months, so no month is missing; the
file's short names just skip 1. After renaming, `PAY_1`, `BILL_AMT1` and
`PAY_AMT1` are all September 2005. (Readers who know the dataset may expect
`PAY_0`, so say this in the report.)

In [3]:
clean = clean.rename(columns={"PAY_0": "PAY_1"})
[c for c in clean.columns if c.startswith("PAY_")][:6]

['PAY_1', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

## Step 3: merge undocumented `EDUCATION` and `MARRIAGE` codes into "others"

The documentation lists `EDUCATION` 1-4 (4 = others) and `MARRIAGE` 1-3
(3 = others). The file also has `EDUCATION` 0, 5, 6 and `MARRIAGE` 0. Since
we don't know what those codes mean, they go into "others".

In [4]:
print("before:", clean["EDUCATION"].value_counts().sort_index().to_dict(),
      clean["MARRIAGE"].value_counts().sort_index().to_dict())

clean["EDUCATION"] = clean["EDUCATION"].replace({0: 4, 5: 4, 6: 4})
clean["MARRIAGE"] = clean["MARRIAGE"].replace({0: 3})

print("after: ", clean["EDUCATION"].value_counts().sort_index().to_dict(),
      clean["MARRIAGE"].value_counts().sort_index().to_dict())

before: {0: 14, 1: 10584, 2: 14027, 3: 4916, 4: 123, 5: 280, 6: 51} {0: 54, 1: 13656, 2: 15962, 3: 323}
after:  {1: 10584, 2: 14027, 3: 4916, 4: 468} {1: 13656, 2: 15962, 3: 377}


## Step 4: `PAY_*` values -2 and 0 are kept as they are

These codes aren't documented either, but they're common, and they are far
less likely to default than 2+ months late. So they are **not** dropped or
recoded here. The file keeps the original numbers. **When modelling, treat
-2, -1 and 0 as categories, not as amounts of delay**: one-hot encode, or
split into "months late" plus a status category. That choice belongs in the
modelling notebook, not in the saved data.

In [5]:
PAY_STATUS = [f"PAY_{i}" for i in range(1, 7)]
print("rows with -2 or 0 in any month:", clean[PAY_STATUS].isin([-2, 0]).any(axis=1).sum(),
      "of", len(clean))

rows with -2 or 0 in any month: 25939 of 29995


## Save

In [6]:
out = PROCESSED / "taiwan-clean.csv"
clean.to_csv(out)          # ID is written as the first column
print(out.relative_to(ROOT), clean.shape)

# Check it reads back identically
back = pd.read_csv(out, index_col="ID")
assert back.equals(clean)

data/processed/taiwan-clean.csv (29995, 24)


To load it elsewhere:

```python
taiwan = pd.read_csv(ROOT / "data" / "processed" / "taiwan-clean.csv", index_col="ID")
```